# Phase 4: Train SimpleCNN on Kaggle

*Made with ❤️ by Calli*

**Where this fits in the pipeline:** data → model → `[train, on a real GPU]` → evaluate → predict

This notebook takes the code we wrote and tested on the laptop and runs it on the **real** dataset with a **GPU**. The code reaches Kaggle as a zip file (no GitHub needed).

---

### One-time setup (and what to repeat when the code changes)

**A. On your laptop: pack the code into a zip.** In the project folder:
```bash
git archive --format=zip --prefix=malaria-cnn/ -o malaria-cnn-code.zip HEAD
```
`git archive` packs the files of your **latest commit** (HEAD) into a zip, inside a `malaria-cnn/` folder. Because it only takes committed files, it never includes `.venv/`, the fake data or old outputs. *Commit first*, otherwise your newest changes aren't in the zip.

**B. On Kaggle: upload the zip as a private dataset.**
1. **Datasets → New Dataset**, upload `malaria-cnn-code.zip`, title it `malaria-cnn-code`, keep it **Private**, click **Create**.
2. *When the code changes later:* make a new zip (step A), open your dataset → **New Version** → upload it.

**C. Set up this notebook.**
1. **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. Right panel → **Session options → Accelerator → GPU** (T4 or P100, either is fine).
3. **Add Input** → add **both**: `cell-images-for-detecting-malaria` (by iarunava) **and** your `malaria-cnn-code` dataset.
4. Internet is **not** needed. Click **Run All**. Expect roughly 15 to 40 minutes in total.

## 1. Settings

Everything this notebook needs to know, in one place. You shouldn't need to change these on Kaggle.

In [ ]:
import shutil
import sys
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")      # attached datasets live here (read-only)
KAGGLE_WORKING = Path("/kaggle/working")  # our writable folder; its contents appear in the Output tab
MODEL_NAME = "simple_cnn"
REQUIRE_GPU = True  # stop early if the GPU isn't switched on; training on CPU would take many hours

## 2. Find our code

Kaggle unpacks uploaded zips into `/kaggle/input/<dataset name>/`, but the exact folder nesting can vary. So instead of guessing a path, we **search** for our `src/train.py` and take the folder above `src/` as the project folder.

In [ ]:
train_scripts = sorted(KAGGLE_INPUT.rglob("src/train.py"))
print("Found:", [str(path) for path in train_scripts])

if len(train_scripts) == 0:
    raise FileNotFoundError("Code not found. Did you Add Input -> your malaria-cnn-code dataset?")
if len(train_scripts) > 1:
    raise RuntimeError("Found the code more than once. Remove the extra input so we know which version to run.")

code_in_input = train_scripts[0].parent.parent  # .../malaria-cnn/src/train.py -> .../malaria-cnn
print("Code folder:", code_in_input)

## 3. Copy the code somewhere writable

`/kaggle/input` is **read-only**, but the first run must *write* `splits/splits.csv` (our saved train/val/test split). So we copy the project into `/kaggle/working/malaria-cnn` and run it from there.

Then we "move into" that folder (`%cd`), so that `python -m src.train` finds the `src` package, and add it to Python's search path so this notebook can import our `config` too.

In [ ]:
PROJECT_DIR = KAGGLE_WORKING / "malaria-cnn"
# dirs_exist_ok=True: if we run this cell twice, overwrite instead of failing.
# A splits.csv created by an earlier run in this session is kept.
shutil.copytree(code_in_input, PROJECT_DIR, dirs_exist_ok=True)

%cd {PROJECT_DIR}
sys.path.insert(0, str(PROJECT_DIR))

from src import config
print("Outputs will be saved to:", config.OUTPUT_DIR)
print("Split file:", config.SPLITS_CSV, "(exists)" if config.SPLITS_CSV.exists() else "(will be created)")

## 4. Check the GPU and record library versions

A GPU makes training 10 to 50 times faster. If this cell stops with an error, switch the accelerator on (setup step C.2) and run again.

We also print the library versions Kaggle uses, so we can pin the same versions in `requirements.txt` on the laptop.

In [ ]:
import matplotlib
import numpy
import pandas
import PIL
import sklearn
import torch
import torchvision
import tqdm

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError("No GPU! Session options -> Accelerator -> GPU, then Run All again.")
else:
    print("WARNING: no GPU, running on CPU (only OK for testing)")

library_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": numpy.__version__,
    "pandas": pandas.__version__,
    "Pillow": PIL.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "tqdm": tqdm.__version__,
}
for library, version in library_versions.items():
    print(f"{library:>13}: {version}")

## 5. Sanity check: can the model memorise 16 real cells?

Same check as on the laptop (`--overfit-batch`), now on **real** images and the GPU. If the loss can't fall close to zero, something is wrong, and the notebook stops here instead of wasting GPU time.

**The very first run is slower:** before anything else, the code finds every image, fingerprints them to remove duplicates, splits them by patient and saves `splits.csv`. Expect about a minute. Read the split table it prints: every split should have both classes, and roughly 70/15/15.

*`{sys.executable}` means "the same Python this notebook uses". `_exit_code` is set by Jupyter after a `!` command: 0 means success.*

In [ ]:
!{sys.executable} -m src.train --model {MODEL_NAME} --data-dir {KAGGLE_INPUT} --overfit-batch

if _exit_code != 0:
    raise RuntimeError("The overfit check failed (or crashed). The output above shows why.")

## 6. Train!

Up to 15 epochs; early stopping usually ends it sooner. One line is printed per epoch:

- **train loss / val loss:** should both go down. Val loss rising while train loss keeps falling = overfitting.
- **val acc / val recall:** share of all validation cells right / share of infected validation cells caught.
- **lr:** the learning rate; it halves when validation loss stalls.

The best epoch's model is saved as `simple_cnn_best.pt`.

In [ ]:
!{sys.executable} -m src.train --model {MODEL_NAME} --data-dir {KAGGLE_INPUT}

if _exit_code != 0:
    raise RuntimeError("Training failed. The output above shows why.")

## 7. Look at the curves

**Left:** train vs validation loss. **Right:** validation accuracy and recall.
Ask yourself: is there a widening gap between the two losses? Where did the best epoch happen, and why did training stop?

In [ ]:
import json

from IPython.display import Image, display

curves_path = config.OUTPUT_DIR / f"{MODEL_NAME}_curves.png"
history_path = config.OUTPUT_DIR / f"{MODEL_NAME}_history.json"

display(Image(filename=str(curves_path)))

with open(history_path) as history_file:
    history = json.load(history_file)
history_table = pandas.DataFrame(history).set_index("epoch").round(4)
print(history_table.to_string())

## 8. Gather the files to download

Everything worth keeping goes into one folder, `/kaggle/working/outputs/`. After the run, open the **Output** tab (right panel, or the notebook's Output page after saving a version) and download:

| File | Where it goes on your laptop |
|---|---|
| `splits.csv` | `splits/splits.csv`, then **commit it**, so every later phase uses the same test patients |
| `simple_cnn_history.json`, `simple_cnn_curves.png` | `results/` (small, worth committing) |
| `simple_cnn_best.pt` | keep it safe, but **don't commit it**. Model weights go to a GitHub Release later |

In [ ]:
shutil.copy(config.SPLITS_CSV, config.OUTPUT_DIR / "splits.csv")

for path in sorted(config.OUTPUT_DIR.iterdir()):
    if path.is_file():
        size_in_mb = path.stat().st_size / 1_000_000
        print(f"{path.name:<30} {size_in_mb:8.2f} MB")

## 9. Summary

Copy **everything this cell prints** and check it for the red flags in PLAN.md Section 7 (e.g. accuracy of 99% or more can mean leakage; a loss stuck at 0.69 means the model isn't learning).

In [ ]:
split_table = pandas.read_csv(config.SPLITS_CSV, dtype={"patient_id": str})
best_epoch = min(history, key=lambda record: record["val_loss"])  # the epoch with the lowest validation loss

print("===== PHASE 4 SUMMARY =====")
print("library versions:", library_versions)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("\nsplit (images / patients / parasitized share):")
for split_name in ["train", "val", "test"]:
    rows = split_table[split_table["split"] == split_name]
    parasitized_share = (rows["label"] == config.LABEL_PARASITIZED).mean()
    print(f"  {split_name:<5} {len(rows):>6} images  {rows['patient_id'].nunique():>4} patients  "
          f"{parasitized_share:.1%} parasitized")
print("\nhistory:")
print(history_table.to_string())
print(f"\nbest epoch: {best_epoch['epoch']} (val loss {best_epoch['val_loss']:.4f}, "
      f"val acc {best_epoch['val_accuracy']:.4f}, val recall {best_epoch['val_recall']:.4f})")
print(f"epochs run: {len(history)} of max {config.MAX_EPOCHS}")

## Done when

Training finished and the curves look sensible.

### Explain it back
1. Looking at *your* curves: did the model overfit? How do you know?
2. Why did training stop when it did?